In [ ]:
#x-y平面に流線を描画するコード（jeans_3d-test26の可視化で使った）

import os
import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import natsort
import matplotlib.animation as animation
from PIL import Image

vtk_dir = os.path.expanduser("~/athenapp/results/〇〇")
output_dir = "./xy_streamline"
os.makedirs(output_dir, exist_ok=True)

# 計算領域
Lbox = 12.0
x_min, x_max = 0.0, 12.0
y_min, y_max = 0.0, 12.0

# 球
Rsphere = 5.0
xc, yc = 6.0, 6.0

# 中央スライス
z_slice = 6.0

# ============================================================
# VTK ファイル取得
# ============================================================
vtk_files = natsort.natsorted([
    os.path.join(vtk_dir, f)
    for f in os.listdir(vtk_dir)
    if f.startswith("Jeans.block") and f.endswith(".vtk")
])

block_dict = {}
for f in vtk_files:
    block_id = int(os.path.basename(f).split('.')[1].replace('block', ''))
    block_dict.setdefault(block_id, []).append(f)

all_blocks_sorted = [block_dict[b] for b in sorted(block_dict.keys())]

# ============================================================
# 球内のみで密度スケール決定
# ============================================================
dens_inside_all = []

for step_files in zip(*all_blocks_sorted):
    pts_all, dens_all = [], []

    for f in step_files:
        grid = pv.read(f)
        sl = grid.slice(normal='z', origin=(0, 0, z_slice))
        pts_all.append(sl.cell_centers().points)
        dens_all.append(sl['dens'])

    pts_all = np.vstack(pts_all)
    dens_all = np.hstack(dens_all)

    r = np.sqrt((pts_all[:, 0]-xc)**2 + (pts_all[:, 1]-yc)**2)
    dens_inside_all.append(dens_all[r <= Rsphere])

dens_inside_all = np.concatenate(dens_inside_all)
vmin = np.percentile(dens_inside_all, 5)
vmax = np.percentile(dens_inside_all, 95)

# ============================================================
# 各タイムステップ描画
# ============================================================
for step_files in zip(*all_blocks_sorted):

    pts_all, dens_all, vx_all, vy_all = [], [], [], []

    for f in step_files:
        grid = pv.read(f)
        sl = grid.slice(normal='z', origin=(0, 0, z_slice))

        centers = sl.cell_centers().points
        dens = sl['dens']
        mom  = sl['mom']

        pts_all.append(centers)
        dens_all.append(dens)
        vx_all.append(mom[:, 0] / dens)
        vy_all.append(mom[:, 1] / dens)

    pts_all = np.vstack(pts_all)
    dens_all = np.hstack(dens_all)
    vx_all = np.hstack(vx_all)
    vy_all = np.hstack(vy_all)

    xs = np.unique(pts_all[:, 0])
    ys = np.unique(pts_all[:, 1])
    nx, ny = len(xs), len(ys)

    dens_2d = np.full((ny, nx), np.nan)
    vx_2d   = np.full((ny, nx), np.nan)
    vy_2d   = np.full((ny, nx), np.nan)

    for i, x in enumerate(xs):
        for j, y in enumerate(ys):
            m = (np.isclose(pts_all[:, 0], x)) & (np.isclose(pts_all[:, 1], y))
            if np.any(m):
                dens_2d[j, i] = dens_all[m][0]
                vx_2d[j, i]   = vx_all[m][0]
                vy_2d[j, i]   = vy_all[m][0]

    X, Y = np.meshgrid(xs, ys)
    r = np.sqrt((X-xc)**2 + (Y-yc)**2)

    dens_in  = np.ma.masked_where(r > Rsphere, dens_2d)
    dens_out = np.ma.masked_where(r <= Rsphere, dens_2d)

    # ========================================================
    # 描画
    # ========================================================
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_aspect('equal')

    # 球外
    ax.pcolormesh(xs, ys, dens_out, color='#0b3c5d')

    # 球内密度
    im = ax.pcolormesh(xs, ys, dens_in,
                       cmap='inferno',
                       norm=LogNorm(vmin=vmin, vmax=vmax))

    # ===== 流線 =====
    ax.streamplot(
        xs, ys,
        vx_2d, vy_2d,
        color='white',
        density=1.2,
        linewidth=0.7,
        arrowsize=0.8
    )

    # 球境界
    theta = np.linspace(0, 2*np.pi, 400)
    ax.plot(xc + Rsphere*np.cos(theta),
            yc + Rsphere*np.sin(theta),
            color='cyan', lw=0.6)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

    filename = os.path.basename(step_files[0])
    num = filename.split('.')[-2]
    ax.set_title(f"Streamlines (γ=1.2, β=0.05)")

    fig.colorbar(im, ax=ax, label="Density (inside sphere)")

    png_file = os.path.join(output_dir, f"stream_{num}.png")
    fig.savefig(png_file, dpi=150, bbox_inches='tight')
    plt.close(fig)

    print(f"Saved {png_file}")